### Овсянникова Майя группа 241, 05.02.2026

### Условие задачи
1. Обработать пропуски (заполнить `Unknown`, **НЕ** удалять)

2. Добавить атрибут – разницу между датой выпуска фильма и датой релиза на Нетфликс (привести к адекватной метрике), найти среднее

3. Разделить датафрейм на два отдельных датафрейма фильмы и сериалы, а атрибут продолжительность сделать числовым (минуты или кол-во сезонов)

4. Разбить атрибут `Country` таким образом, чтобы каждая строка с фильмом/сериалом была продублирована столько раз, сколько стран перечислено в этом фильме, по одной стране на каждую запись

5. ТОП-5 чего-либо (ТОП-5 самых продолжительных фильмов, ТОП-5 самых популярных жанров, придумать еще два ТОП-5 своих)

### Теория

Я загрузила и подготовила данные, заполнив пропуски и преобразовав даты. Затем создала новый атрибут, показывающий задержку между выпуском контента и его появлением на платформе. Данные были разделены на фильмы и сериалы с преобразованием продолжительности в числовой формат. Далее я разбила строки с несколькими значениями атрибута `Country` на отдельные записи. После чего составила несколько топ-5 таблиц по четырем параметрам.

### Документация

Используемые библиотеки:
- pandas – основная библиотека для работы с табличными данными

Основные методы pandas:
- `pd.read_csv()` – загрузка данных из csv файла
- `df.fillna()` – заполнение пропущенных значений указанной строкой
- `pd.to_datetime()` – преобразование строк в формат даты (`errors='coerce'` – преобразует некорректные значения в `NaT`)
- `df.copy()` – создание копии DataFrame для безопасного изменения
- `.dt.year` – извлечение года из даты
- `.mean()` – вычисление среднего значения

Методы обработки строк:
- `.str.split()` – разделение строк по разделителю (`str[0]` – получение первого элемента после разделения)
- `.str.split(', ')` – разделение строк по запятой с пробелом
- `pd.to_numeric()` – преобразование строк в числовой формат (`errors='coerce'` – преобразует некорректные значения в `NaN`)

Методы фильтрации и выбора данных:
- `df[df['column'] == value]` – фильтрация по условию
- `.nlargest()` – получение n наибольших значений по столбцу

Методы для работы со списками в ячейках:
- `.explode()` – преобразование списков в отдельных строках

Методы агрегации и статистики:
- `.value_counts()` – подсчет частоты уникальных значений
- `len()` – подсчет количества элементов

### Решение

#### Загрузка и предварительная обработка данных

In [187]:
import pandas as pd

df = pd.read_csv('netflix.csv')
df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


#### 1. Обработать пропуски – заполнить `Unknown`

In [188]:
df = df.fillna('Unknown')
df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


#### 2. Добавить атрибут – разницу между датой выпуска фильма и датой релиза на Нетфликс (привести к адекватной метрике), найти среднее

In [189]:
df["date_added"] = pd.to_datetime(df["date_added"], errors = "coerce")
df["release_add_diff"] = df["date_added"].dt.year - df["release_year"]

df.head(3)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,release_add_diff
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",1.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",0.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,0.0


In [190]:
avg_release_add_diff = df['release_add_diff'].mean().round(4)
print("Cредняя разница между датой выпуска фильма и датой релиза на Нетфликс (лет):", avg_release_add_diff)

Cредняя разница между датой выпуска фильма и датой релиза на Нетфликс (лет): 4.6909


#### 3. Разделить датафрейм на два отдельных датафрейма фильмы и сериалы, а атрибут продолжительность сделать числовым (минуты или кол-во сезонов)

In [191]:
df_movies = df[df["type"] == "Movie"].copy()
df_movies['duration_m'] = pd.to_numeric(df_movies["duration"].str.split().str[0], errors='coerce')
df_movies.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,release_add_diff,duration_m
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",1.0,90.0
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",Unknown,2021-09-24,2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...,0.0,91.0
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...",2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",28.0,125.0


In [192]:
df_tvshows = df[df["type"] == "TV Show"].copy()
df_tvshows['duration_s'] = pd.to_numeric(df_tvshows["duration"].str.split().str[0], errors='coerce')
df_tvshows.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,release_add_diff,duration_s
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",0.0,2
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,0.0,1
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",0.0,1


In [193]:
print("Количество Movies:", len(df_movies))
print("Количество TV Shows:", len(df_tvshows))

Количество Movies: 6131
Количество TV Shows: 2676


#### 4. Разбить атрибут `Country` таким образом, чтобы каждая строка с фильмом/сериалом была продублирована столько раз, сколько стран перечислено в этом фильме, по одной стране на каждую запись

In [194]:
df_countries = df.copy()
df_countries['country'] = df_countries['country'].str.split(',')
df_countries = df_countries.explode('country')

df_countries.head(15)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,release_add_diff
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",1.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",0.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,0.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",0.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,0.0
5,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, H...",Unknown,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...,0.0
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",Unknown,2021-09-24,2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...,0.0
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...",United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",28.0
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...",Ghana,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",28.0
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...",Burkina Faso,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",28.0


In [195]:
print("До разбиения Country:", len(df), "строк")
print("После разбиения Country:", len(df_countries), "строк")

До разбиения Country: 8807 строк
После разбиения Country: 10850 строк


#### 5. Четыре ТОП-5

##### **–** ТОП-5 самых продолжительных фильмов

In [196]:
top5_duration_movies = df_movies.nlargest(5, 'duration_m')[['title', 'duration_m']]
print(top5_duration_movies)

                            title  duration_m
4253   Black Mirror: Bandersnatch       312.0
717   Headspace: Unwind Your Mind       273.0
2491       The School of Mischief       253.0
2487               No Longer kids       237.0
2484           Lock Your Girls In       233.0


##### **–** ТОП-5 самых популярных жанров

In [197]:
df_genres = df.copy()
df_genres["listed_in"] = df_genres["listed_in"].str.split(", ")
df_genres = df_genres.explode("listed_in")

top5_popular_genres = df_genres["listed_in"].value_counts().head(5)
print(top5_popular_genres)


listed_in
International Movies      2752
Dramas                    2427
Comedies                  1674
International TV Shows    1351
Documentaries              869
Name: count, dtype: int64


##### **–** ТОП-5 самых длинных ТВ шоу по количеству сезонов

In [198]:
top5_longest_tvshows = df_tvshows.nlargest(5, 'duration_s')[['title', 'duration_s']]
print(top5_longest_tvshows)

                       title  duration_s
548           Grey's Anatomy          17
2423            Supernatural          15
4798                    NCIS          15
1354               Heartland          13
4220  COMEDIANS of the world          13


##### **–** ТОП-5 лет с наибольшим количеством добавленного контента

In [199]:
top5_years = df["date_added"].dt.year.value_counts().head(5)
print(top5_years)

date_added
2019.0    1999
2020.0    1878
2018.0    1625
2021.0    1498
2017.0    1164
Name: count, dtype: int64


#### Вывод
**Задание 1:**
Все имеющиеся пропуски заполнены значенем `Unknown` (без удаления)

**Задание 2:** 
Рассчитана задержка при добавлении контента на платформу – разница между датой выпуска фильма и датой релиза на Нетфликс. Получено среднее значение задержки = `4.6909` года

**Задание 3:**
Фильмы и ТВ шоу разделены на разные датасеты, посчитано числовое знание продолжительности каждой единицы контента. Количество Movies: `6131`, количество TV Shows: `2676`

**Задание 4:**
Атрибут `Country` разбит построчно при наличии нескольких стран. До разбиения Country: `8807` строк, после разбиения Country: `10850` строк

**Задание 5:**
Получены четыре ТОП-5 значения по датасету

*ТОП-5 самых продолжительных фильмов* – Black Mirror: Bandersnatch, Headspace: Unwind Your Mind, The School of Mischief, No Longer kids, Lock Your Girls In

*ТОП-5 самых популярных жанров* – International Movies, Dramas, Comedies, International TV Shows, Documentaries

*ТОП-5 самых длинных ТВ шоу по количеству сезонов* – Grey's Anatomy, Supernatural, NCIS, Heartland, COMEDIANS of the world

*ТОП-5 с наибольшим количеством добавленного контента* – 1999, 2020, 2018, 2021, 2017